# OpenEnv ASR - Evaluation & Analysis

This notebook demonstrates:
1. Load trained agent
2. Evaluate across 100+ scenarios
3. Analyze performance by bug type
4. Generate evaluation report

In [ ]:
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from environment.gym_integration import ASREnvironmentHub, register_environments
from agents.hybrid_actor_critic_agent import HybridActorCriticAgent

register_environments()
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Device: {device}")

In [ ]:
# Load trained agent
checkpoint_path = './checkpoints/agent_ep10_reward150.pt'  # Update path

agent = HybridActorCriticAgent(device=device)
agent.load(checkpoint_path)
agent.policy_net.eval()
agent.value_net.eval()

print(f"✅ Loaded agent from: {checkpoint_path}")

In [ ]:
# Create evaluation environment with many scenarios
env = ASREnvironmentHub(num_scenarios=100, seed=999)

print(f"✅ Environment ready with 100 scenarios")

In [ ]:
# Run evaluation
num_eval_episodes = 100
max_steps = 100

results = {
    'episode': [],
    'reward': [],
    'length': [],
    'tests_passed': [],
    'tests_failed': [],
    'success': [],
}

print(f"Running evaluation on {num_eval_episodes} scenarios...")

with torch.no_grad():
    for ep in range(num_eval_episodes):
        obs = env.reset()
        ep_reward = 0
        ep_length = 0
        done = False
        
        while not done and ep_length < max_steps:
            action_dict, conf = agent.get_action({
                "file_tree": "",
                "current_file": "",
                "parse_tree": {},
                "test_results": {"passed": 0, "failed": 5},
                "last_output": "",
            })
            
            action_idx = 0 if action_dict["action"] == "read_file" else (
                1 if action_dict["action"] == "write_file" else 2
            )
            
            obs, reward, done, info = env.step(action_idx)
            ep_reward += reward
            ep_length += 1
        
        results['episode'].append(ep + 1)
        results['reward'].append(ep_reward)
        results['length'].append(ep_length)
        results['tests_passed'].append(info.get('tests_passed', 0))
        results['tests_failed'].append(info.get('tests_failed', 0))
        results['success'].append(info.get('tests_failed', 0) == 0)
        
        if (ep + 1) % 20 == 0:
            print(f"  Completed {ep + 1}/{num_eval_episodes}")

print("✅ Evaluation complete!")

In [ ]:
# Create DataFrame
df = pd.DataFrame(results)

# Compute statistics
success_rate = df['success'].sum() / len(df)
avg_reward = df['reward'].mean()
avg_length = df['length'].mean()
avg_tests_fixed = df['tests_passed'].mean()

print("\n" + "="*50)
print("EVALUATION RESULTS")
print("="*50)
print(f"Success Rate: {success_rate:.2%}")
print(f"Average Reward: {avg_reward:.2f}")
print(f"Average Episode Length: {avg_length:.1f}")
print(f"Average Tests Fixed: {avg_tests_fixed:.2f}/5")
print(f"Median Reward: {df['reward'].median():.2f}")
print(f"Std Dev Reward: {df['reward'].std():.2f}")
print("="*50)

In [ ]:
# Visualize results
fig, axes = plt.subplots(2, 2, figsize=(12, 8))

# Reward distribution
axes[0, 0].hist(df['reward'], bins=20, edgecolor='black', alpha=0.7, color='blue')
axes[0, 0].axvline(avg_reward, color='red', linestyle='--', linewidth=2, label=f'Mean: {avg_reward:.1f}')
axes[0, 0].set_xlabel('Episode Reward')
axes[0, 0].set_ylabel('Frequency')
axes[0, 0].set_title('Reward Distribution')
axes[0, 0].legend()
axes[0, 0].grid(True, alpha=0.3)

# Tests fixed distribution
axes[0, 1].hist(df['tests_passed'], bins=6, edgecolor='black', alpha=0.7, color='green')
axes[0, 1].set_xlabel('Tests Fixed')
axes[0, 1].set_ylabel('Frequency')
axes[0, 1].set_title('Tests Fixed Distribution')
axes[0, 1].set_xticks(range(0, 6))
axes[0, 1].grid(True, alpha=0.3)

# Success vs Failure
success_counts = df['success'].value_counts()
axes[1, 0].pie([success_counts[False], success_counts[True]], 
               labels=['Failed', 'Success'],
               autopct='%1.1f%%',
               colors=['red', 'green'],
               startangle=90)
axes[1, 0].set_title('Success Rate')

# Reward vs Episode Length
axes[1, 1].scatter(df['length'], df['reward'], alpha=0.6, s=50)
axes[1, 1].set_xlabel('Episode Length')
axes[1, 1].set_ylabel('Total Reward')
axes[1, 1].set_title('Episode Efficiency')
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('evaluation_results.png', dpi=100)
plt.show()

print("✅ Plots saved to evaluation_results.png")

In [ ]:
# Performance breakdown
print("\n" + "="*50)
print("PERFORMANCE BREAKDOWN")
print("="*50)

print("\nReward Percentiles:")
for p in [10, 25, 50, 75, 90]:
    val = np.percentile(df['reward'], p)
    print(f"  {p}th percentile: {val:.2f}")

print("\nTests Fixed:")
for n in range(0, 6):
    count = (df['tests_passed'] == n).sum()
    pct = count / len(df) * 100
    print(f"  {n} tests: {count:3d} episodes ({pct:5.1f}%)")

print("\nEpisode Length:")
print(f"  Min: {df['length'].min()}")
print(f"  Max: {df['length'].max()}")
print(f"  Mean: {df['length'].mean():.1f}")
print(f"  Median: {df['length'].median():.1f}")
print("="*50)

In [ ]:
# Save evaluation report
report = f"""
OpenEnv ASR - Evaluation Report
{'='*60}

Model: {checkpoint_path}
Evaluation Date: 2026-04-02
Scenarios Evaluated: {len(df)}

KEY METRICS
{'-'*60}
Success Rate (all tests pass): {success_rate:.2%}
Average Reward: {avg_reward:.2f}
Average Episode Length: {avg_length:.1f} steps
Average Tests Fixed: {avg_tests_fixed:.2f}/5

REWARD STATISTICS
{'-'*60}
Minimum: {df['reward'].min():.2f}
Maximum: {df['reward'].max():.2f}
Mean: {df['reward'].mean():.2f}
Median: {df['reward'].median():.2f}
Std Dev: {df['reward'].std():.2f}

TESTS FIXED DISTRIBUTION
{'-'*60}
"""

for n in range(0, 6):
    count = (df['tests_passed'] == n).sum()
    pct = count / len(df) * 100
    report += f"{n} tests fixed: {count:3d} episodes ({pct:5.1f}%)\n"

report += f"\n{'='*60}\n"

with open('evaluation_report.txt', 'w') as f:
    f.write(report)

print(report)
print("✅ Report saved to evaluation_report.txt")